# Chapitre 3 — Le Cloud pour la data

**Durée estimée : 8-10 heures**

---

## Objectifs d'apprentissage

À la fin de ce chapitre, vous serez capable de :

1. **Expliquer** les avantages du cloud computing et identifier quand l'utiliser plutôt qu'un environnement local
2. **Configurer** un compte AWS et gérer les credentials de manière sécurisée
3. **Lire et écrire** des données depuis/vers AWS S3 en utilisant boto3 et pandas
4. **Comparer** les équivalences entre AWS, GCP et Azure pour le stockage de données

---


## 3.7 Lire depuis S3 avec Python

---

### Introduction : Comprendre les flux de lecture depuis S3

**Comment récupérer votre donnée stockée dans le Cloud ?**

Comme pour l'écriture, la lecture depuis S3 peut se faire de deux manières fondamentales :

***1. La lecture "Physique" (Téléchargement sur disque)***

On télécharge le fichier depuis S3 vers notre disque dur local, puis on l'ouvre.

* **L'enjeu :** Utile quand on veut conserver le fichier localement ou le partager.
* **Le risque :** Opération lente (I/O disque) et nécessite de l'espace de stockage local.

***2. La lecture "Streaming" (Directement en RAM)***

On lit le fichier directement dans la mémoire vive sans jamais le sauvegarder sur le disque local.

* **L'enjeu :** Plus rapide, plus propre (aucun fichier temporaire), idéal pour les environnements Cloud (Lambda, Fargate).
* **Le risque :** La RAM est limitée. Si le fichier est trop volumineux, on risque un Memory Error.

---

***Ce que nous allons apprendre :***

Dans ce cours, nous allons voir comment lire des données depuis S3 avec les trois outils majeurs :

1. **Pandas + s3fs :** Pour la simplicité maximale (une ligne de code).
2. **Boto3 :** Pour le contrôle total et la compréhension des fondations.
3. **AWS Wrangler :** Pour les fonctionnalités avancées (lecture de datasets partitionnés).

---

### Installation des dépendances

In [ ]:
# Installation (exécuter une seule fois)
!pip3 install boto3 pandas s3fs pyarrow awswrangler

---

### Outil 1 : Pandas natif avec URI S3

La méthode la plus simple — pandas gère tout en arrière-plan grâce à `s3fs` :

In [ ]:
import pandas as pd

# Lecture directe d'un CSV depuis S3 (nécessite s3fs installé)
df = pd.read_csv("s3://mon-bucket/raw/ventes.csv")

print("Avec s3fs installé, pandas peut lire directement depuis S3 !")
print("Syntaxe : pd.read_csv('s3://bucket/chemin/fichier.csv')")
print(f"\nNombre de lignes chargées : {len(df)}")
df.head()

In [ ]:
# Lecture d'un fichier Parquet (format recommandé)
df_parquet = pd.read_parquet("s3://mon-bucket/processed/ventes.parquet")

# Pandas utilise s3fs pour la connexion et pyarrow pour décoder le format Parquet.
# Le Parquet est plus rapide à lire car il est compressé et stocke les types de données.

print("Lecture Parquet terminée !")
df_parquet.head()

**Avantage** : Simplicité maximale, une seule ligne de code.

**Limitation** : Nécessite la librairie `s3fs` installée, qui peut créer des conflits de dépendances dans certains environnements.

**Quand l'utiliser** : Notebooks (SageMaker/Jupyter) et prototypage rapide.

---

### Outil 2 : Boto3

In [ ]:
import boto3

# Créer le client S3 (utilise ~/.aws/credentials automatiquement)
s3_client = boto3.client('s3')

# Action : Initialisation du client S3.
# Explication : On demande à la bibliothèque boto3 de créer une interface de bas niveau avec le service S3. 
# Le "client" est l'outil qui va nous permettre d'appeler les API officielles d'AWS (comme get_object, download_file, etc.).

---

#### Télécharger un fichier physiquement

C'est comme recevoir un colis par la poste.

- Le fichier est téléchargé depuis S3 et écrit physiquement sur ton disque dur.

- Le flux : Ton code dit à AWS : "Envoie-moi ce fichier qui s'appelle ventes.csv et sauvegarde-le ici sur mon disque".

- Usage type : Tu veux conserver le fichier localement, le partager, ou l'utiliser avec un outil qui ne supporte pas le streaming.

In [ ]:
# Télécharger un fichier S3 vers le disque local
s3_client.download_file(
    Bucket="mon-bucket",           # Nom du bucket source
    Key="raw/ventes.csv",          # Chemin (clé) du fichier dans S3
    Filename="ventes_local.csv"    # Nom du fichier local de destination
)

print("boto3.download_file() télécharge un fichier S3 vers le disque local")

# Mécanisme : Boto3 récupère le fichier par morceaux (chunks) et l'écrit sur le disque.
# Avantages :
# - Gestion automatique des gros fichiers (Multi-part download)
# - Simplicité : Le fichier est prêt à être utilisé localement
# Inconvénient : Nécessite de l'espace disque et prend du temps (I/O).

In [ ]:
import pandas as pd

# Maintenant on peut lire le fichier téléchargé
df_local = pd.read_csv("ventes_local.csv")
print(f"Fichier chargé avec {len(df_local)} lignes")
df_local.head()

---

#### Lecture en streaming (directement en mémoire)

C'est la méthode de lecture directe en mémoire. Au lieu de télécharger un fichier sur ton ordinateur pour l'ouvrir ensuite, tu crées un "tunnel" entre S3 et ton script Python.

- Les données passent directement de S3 à ta RAM.
- Le fichier n'existe jamais physiquement sur ta machine.
- Avantages : 
    - **Vitesse** : On évite les temps d'écriture sur disque.
    - **Sécurité** : Aucune donnée sensible ne reste sur le disque après l'exécution du script.
    - **Propreté** : Pas de fichiers temporaires à gérer.

In [ ]:
import boto3
import pandas as pd
from io import BytesIO  # AWS renvoie les données de S3 sous forme de données binaires (des octets). 
                        # BytesIO est le buffer adapté pour traiter ces flux d'octets en mémoire.

s3_client = boto3.client("s3")

# Récupérer l'objet en mémoire (sans télécharger sur disque)
response = s3_client.get_object(Bucket="mon-bucket", Key="raw/ventes.csv")
# Action : On demande à S3 de nous donner l'accès à l'objet.
# Important : Cette ligne ne télécharge pas encore tout le contenu. 
# Elle récupère une "réponse" qui contient des métadonnées (taille, type de fichier) 
# et surtout un StreamingBody (un flux de données ouvert).

df = pd.read_csv(BytesIO(response["Body"].read()))
# C'est ici que tout se passe. Décomposons de l'intérieur vers l'extérieur :
# 1. response["Body"].read() : On lit l'intégralité du contenu binaire du fichier depuis S3 vers la RAM.
# 2. BytesIO(...) : On transforme ces données brutes en un "fichier virtuel" que Python peut manipuler.
# 3. pd.read_csv(...) : Pandas croit qu'il lit un fichier sur le disque, alors qu'il lit directement dans la mémoire vive.

print("Le streaming évite de créer des fichiers temporaires sur le disque")
print(f"DataFrame chargé avec {len(df)} lignes")
df.head()

#### Lecture streaming pour fichiers Parquet

In [ ]:
import boto3
import pandas as pd
from io import BytesIO

s3_client = boto3.client("s3")

# Lecture Parquet en streaming
response = s3_client.get_object(Bucket="mon-bucket", Key="processed/ventes.parquet")
df_parquet = pd.read_parquet(BytesIO(response["Body"].read()))

print("Lecture Parquet en streaming terminée !")
print(f"DataFrame chargé avec {len(df_parquet)} lignes")
df_parquet.head()

**Avantage** : Contrôle total, fonctionne partout où boto3 est installé, idéal pour AWS Lambda.

**Quand l'utiliser** : Environnements avec contraintes de dépendances, ou quand vous avez besoin d'accéder aux métadonnées de l'objet.

#### Supprimer un objet de S3

In [ ]:
# Suppression de l'objet
s3_client.delete_object(
    Bucket='nom-de-votre-bucket',
    Key='dossier/mon-fichier.txt'
)

---

### Outil 3 : AWS SDK for pandas (awswrangler)

La solution "enterprise-grade" recommandée par AWS :

Le "plus" : Tu n'as pas besoin de créer de client S3, ni de gérer de buffer. Tu donnes l'adresse S3 (format URI), et Wrangler s'occupe de tout techniquement en arrière-plan.

C'est particulièrement utile pour lire des **datasets partitionnés** (plusieurs fichiers organisés en dossiers).

In [ ]:
import awswrangler as wr

# Lecture simple d'un fichier CSV
df = wr.s3.read_csv("s3://mon-bucket/raw/ventes.csv")

print("Lecture CSV avec AWS Wrangler")
df.head()

In [ ]:
# Lecture d'un fichier Parquet
df_parquet = wr.s3.read_parquet("s3://mon-bucket/processed/ventes.parquet")

print("Lecture Parquet avec AWS Wrangler")
df_parquet.head()

In [ ]:
# La puissance de Wrangler : Lire un dataset partitionné entier
# Si tu as des fichiers organisés comme :
#   s3://mon-bucket/processed/ventes/annee=2023/mois=01/file1.parquet
#   s3://mon-bucket/processed/ventes/annee=2023/mois=02/file2.parquet
#   s3://mon-bucket/processed/ventes/annee=2024/mois=01/file3.parquet

df_all = wr.s3.read_parquet("s3://mon-bucket/processed/ventes/")
# Wrangler lit TOUS les fichiers Parquet du dossier et les combine automatiquement !
# Les colonnes de partition (annee, mois) sont ajoutées au DataFrame.

print("Lecture d'un dataset partitionné complet")
print(f"Total : {len(df_all)} lignes")
df_all.head()

In [ ]:
# Lecture avec filtre sur les partitions (optimisation des coûts !)
# On ne lit QUE les données de janvier 2024

df_filtered = wr.s3.read_parquet(
    "s3://mon-bucket/processed/ventes/",
    partition_filter=lambda x: x["annee"] == "2024" and x["mois"] == "01"
)

# Avantage économique : Au lieu de télécharger 10 Go de données,
# on ne télécharge que les ~500 Mo du mois concerné.
# Cela réduit le temps de traitement ET la facture AWS !

print("Lecture avec filtre sur les partitions")
print(f"Lignes chargées : {len(df_filtered)}")
df_filtered.head()

**Avantage** : Fonctionnalités avancées (lecture de datasets partitionnés, intégration Glue Catalog, filtrage intelligent).

**Quand l'utiliser** : Data Lakes, gros volumes de données, pipelines complexes.

**Installation** : `pip install awswrangler`

*(Source : [AWS SDK for pandas Documentation](https://aws-sdk-pandas.readthedocs.io/en/stable/tutorials/003%20-%20Amazon%20S3.html))*

---

### Comparaison des méthodes de lecture

| Méthode | Type de lecture | Formats supportés | Utilité principale | Avantage clé | Quand l'utiliser ? |
|---------|-----------------|-------------------|--------------------|--------------|-----------------------|
| Pandas (+ s3fs) | Mémoire (streaming) | CSV, Parquet, JSON | Analyse de données rapide | Simplicité maximale : Une seule ligne de code avec l'URI s3:// | Notebooks (SageMaker/Jupyter) et prototypage rapide |
| Boto3 | Physique (download_file) & Mémoire (get_object) | Tous (CSV, JSON, Images, Zip...) | Interaction native avec l'API AWS | Léger & Sans dépendance : Pas besoin d'installer de grosses librairies | AWS Lambda ou scripts nécessitant un contrôle fin |
| AWS Wrangler | Mémoire (optimisé) | CSV, Parquet, JSON, Excel... | Data Engineering de niveau entreprise | Puissant & Intelligent : Lecture de datasets partitionnés avec filtres | Data Lakes, gros volumes et pipelines complexes |

---

### Le schéma de décision

| Besoin | Méthode recommandée | Pourquoi ? |
|--------|---------------------|------------|
| Lire un fichier rapidement en notebook | `pd.read_csv("s3://...")` | Simple et direct, une ligne de code. |
| Conserver le fichier sur le disque local | `boto3.download_file()` | Le fichier reste disponible après l'exécution. |
| Lire sans écrire sur le disque (Lambda, Fargate) | `boto3.get_object()` + `BytesIO` | Pas de fichier temporaire, idéal pour le serverless. |
| Lire un dataset partitionné (Data Lake) | `wr.s3.read_parquet()` | Combine automatiquement tous les fichiers, supporte les filtres. |
| Lire seulement une partie des données | `wr.s3.read_parquet()` avec `partition_filter` | Économise du temps et de l'argent. |

---

### Lister les fichiers d'un bucket

Avant de lire, il est souvent utile de voir ce qui existe dans un bucket :

In [ ]:
import boto3

s3_client = boto3.client("s3")

# Lister les objets d'un bucket (avec préfixe optionnel)
response = s3_client.list_objects_v2(
    Bucket="mon-bucket",
    Prefix="raw/"  # Optionnel : filtre par "dossier"
)

print("Fichiers dans s3://mon-bucket/raw/ :")
for obj in response.get("Contents", []):
    print(f"  - {obj['Key']} ({obj['Size']} bytes)")

In [ ]:
# Avec AWS Wrangler (plus simple)
import awswrangler as wr

fichiers = wr.s3.list_objects("s3://mon-bucket/raw/")

print("Fichiers listés avec Wrangler :")
for f in fichiers:
    print(f"  - {f}")

---

### Bonnes pratiques de lecture

| Pratique | Raison |
|----------|--------|
| Privilégier Parquet | 3-10x plus rapide à lire que CSV, types de données préservés. |
| Utiliser le streaming en production | Évite les fichiers temporaires et les problèmes d'espace disque. |
| Filtrer les partitions si possible | Réduit le temps de lecture et les coûts de transfert AWS. |
| Vérifier la taille avant de charger | Un fichier de 50 Go ne tiendra pas en RAM ! |
| Utiliser `chunksize` pour les gros CSV | `pd.read_csv(..., chunksize=10000)` lit par morceaux. |

---

### Lecture par morceaux (chunking) pour les gros fichiers

Quand un fichier est trop gros pour tenir en mémoire :

In [ ]:
import pandas as pd

# Lecture par morceaux de 10 000 lignes
chunks = pd.read_csv("s3://mon-bucket/raw/gros_fichier.csv", chunksize=10000)

# Traiter chaque morceau un par un
for i, chunk in enumerate(chunks):
    print(f"Traitement du chunk {i+1} : {len(chunk)} lignes")
    # Faire des traitements sur le chunk...
    # chunk.to_parquet(f"s3://mon-bucket/processed/chunk_{i}.parquet")
    
    if i >= 2:  # Pour la démo, on s'arrête après 3 chunks
        print("... (arrêt de la démo)")
        break

---

### ✍️ Exercice 3.7 : Pipeline complet S3 (20 min)

Créez un mini-pipeline qui :
1. Lit un CSV depuis S3 (raw/)
2. Fait un nettoyage simple (supprime les doublons)
3. Sauvegarde en Parquet dans S3 (processed/)

In [ ]:
import pandas as pd

def pipeline_s3(bucket, input_key, output_key):
    """
    Pipeline complet : lecture S3 -> nettoyage -> écriture S3.
    
    Args:
        bucket: Nom du bucket
        input_key: Chemin du fichier source (ex: 'raw/clients.csv')
        output_key: Chemin de sortie (ex: 'processed/clients_clean.parquet')
    """
    # 1. Lecture depuis S3
    print(f"Lecture de s3://{bucket}/{input_key}...")
    df = pd.read_csv(f"s3://{bucket}/{input_key}")
    print(f"   Lignes avant nettoyage : {len(df)}")
    
    # 2. Nettoyage (suppression doublons)
    df_clean = df.drop_duplicates()
    lignes_supprimees = len(df) - len(df_clean)
    print(f"   Doublons supprimés : {lignes_supprimees}")
    print(f"   Lignes après nettoyage : {len(df_clean)}")
    
    # 3. Sauvegarde vers S3 en Parquet
    print(f"Sauvegarde vers s3://{bucket}/{output_key}...")
    df_clean.to_parquet(f"s3://{bucket}/{output_key}")
    
    print(f"Pipeline terminé !")
    return df_clean

# Exemple d'utilisation :
# df_result = pipeline_s3(
#     bucket="mon-data-lake",
#     input_key="raw/clients.csv",
#     output_key="processed/clients_clean.parquet"
# )

print("Fonction pipeline_s3() définie et prête à utiliser")

---

### Récapitulatif : Écriture vs Lecture

| Opération | Boto3 (Physique) | Boto3 (Mémoire) | Pandas natif | AWS Wrangler |
|-----------|------------------|-----------------|--------------|---------------|
| **Écrire** | `upload_file()` | `put_object()` + Buffer | `df.to_csv("s3://...")` | `wr.s3.to_csv()` |
| **Lire** | `download_file()` | `get_object()` + BytesIO | `pd.read_csv("s3://...")` | `wr.s3.read_csv()` |

---

---

### CSV vs Parquet : Le choix qui change tout

| Critère | CSV | Parquet |
|---------|-----|---------|
| Lisibilité humaine | ✅ Oui (texte) | ❌ Non (binaire) |
| Taille fichier | 100% | **15-30%** (compression) |
| Vitesse de lecture | Lente | **2x plus rapide** |
| Coût S3/Athena | Élevé | **Très réduit** |
| Types de données | Perdus | ✅ Préservés |

**Chiffres clés** : Selon AWS, Parquet est "2x plus rapide à décharger et consomme jusqu'à 6x moins de stockage sur S3 par rapport aux formats texte". Une étude sur AWS Athena montre que les requêtes sur Parquet coûtent **$3.65/an** contre **$2,000/an** pour les mêmes données en CSV. *(Source : [AWS Athena Parquet vs CSV](https://www.linkedin.com/pulse/aws-athena-parquet-vs-csv-ahmed-fayed))*

> 🤔 **Question Socratique** : Si Parquet est tellement supérieur, pourquoi le CSV existe-t-il encore ? Dans quels contextes précis le CSV reste-t-il le meilleur choix ?

> 🤔 **Réponse** :

**1. La Compatibilité Universelle (Le "Langage Commun")**

Le CSV est **lisible par l'humain**. Vous pouvez l'ouvrir avec :

* Un simple bloc-notes (Notepad).
* Excel ou Google Sheets sans aucune importation complexe.
* N'importe quel langage de programmation, même très ancien.

> **Contexte idéal :** Envoyer un export de données à un client ou à un collègue du service marketing qui n'utilise pas Python ou AWS Athena.

**2. Les Petits Volumes de Données**

Le format Parquet est un format "lourd" en métadonnées (il stocke les schémas, les types, etc.).

* Pour un fichier de **10 lignes**, un fichier Parquet sera plus volumineux qu'un CSV à cause de ces informations structurelles.
* Le gain de performance de Parquet ne se ressent qu'à partir de plusieurs dizaines de milliers de lignes.

> **Contexte idéal :** Fichiers de configuration, petits référentiels (ex: liste des codes postaux), ou tests rapides.

**3. La Phase de Collecte (Ingestion)**

La plupart des applications tierces (CRM, banques, capteurs IoT) et les APIs web génèrent du texte brut.

* Transformer ces flux en Parquet **pendant** qu'on les reçoit demande de la puissance de calcul.
* Il est souvent plus simple et sécurisé de stocker le flux entrant tel quel en CSV dans le dossier `raw/`.

> **Contexte idéal :** Étape 1 de votre pipeline (Ingestion), avant tout traitement.

---

**En résumé**

* **CSV :** Pour l'échange, la lisibilité humaine et la simplicité.
* **Parquet :** Pour l'analyse massive, la performance et l'économie de coûts sur le Cloud.

---